# V3 Silver runner: Silver V2 to reviewed Silver V3

Use a verified Silver V2 snapshot as the V3 Bronze-equivalent input. First inspect keyword/overlap evidence and the deterministic News pilot; only then freeze the label guide, create the full decision file, and review every News row. Silver V3 publication is write-once and requires explicit human approval for each decision.

The local LLM is optional and emits unapproved suggestions only. All publish/model flags default to False. Silver V3 keeps the text and all non-News labels unchanged.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = os.environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_v3_project")
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("Colab checkout has uncommitted changes; resolve them first")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError("Project directory contains non-Git files")
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=PROJECT_DIR, check=True)
SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing project Python sources: {SRC_DIR}")
sys.path[:] = [part for part in sys.path if part != str(SRC_DIR)]
sys.path.insert(0, str(SRC_DIR))
prefix = "vietnamese_topic_classification"
for name in sorted([name for name in sys.modules if name == prefix or name.startswith(prefix + ".")], key=len, reverse=True):
    del sys.modules[name]
importlib.invalidate_caches()

from vietnamese_topic_classification.analysis.news_v3 import build_keyword_report, build_prompt, sample_news_records
from vietnamese_topic_classification.config import get_v3_domain_labels
from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake
from vietnamese_topic_classification.data.gold import GoldConfig, GoldReader, GoldV3Pipeline
from vietnamese_topic_classification.data.silver import SilverReader, SilverV3Pipeline

print("Source revision:", subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
print("Mounted Data Lake:", DATA_LAKE_ROOT)

In [ ]:
SILVER_V2_REPORT_NAME = None  # Set an exact filename if multiple reports exist.
reports_dir = DATA_LAKE_ROOT / "04_reports" / "silver" / "v2"
available = sorted(path.name for path in reports_dir.glob("silver_quality_v2_*.json"))
print("Silver V2 reports:", available)
if SILVER_V2_REPORT_NAME is None:
    if len(available) != 1:
        raise ValueError(f"Choose one exact SILVER_V2_REPORT_NAME; found {len(available)}")
    SILVER_V2_REPORT_NAME = available[0]
if Path(SILVER_V2_REPORT_NAME).name != SILVER_V2_REPORT_NAME:
    raise ValueError("Report name must be a filename, not a path")
silver_v2 = SilverReader(DATA_LAKE_ROOT, SILVER_V2_REPORT_NAME)
verified_v2 = SilverV3Pipeline(silver_v2).verify_source()
print("Silver V2 + Silver V1 lineage verified:", verified_v2)

In [ ]:
v3_reports = DATA_LAKE_ROOT / "04_reports" / "silver" / "v3"
source_suffix = f"{verified_v2['sha256'][:12]}_{hashlib.sha256(silver_v2.report_path.read_bytes()).hexdigest()[:8]}"
KEYWORD_REPORT = v3_reports / f"news_keyword_analysis_v3_{source_suffix}.json"
if KEYWORD_REPORT.exists():
    keyword_report = json.loads(KEYWORD_REPORT.read_text(encoding="utf-8"))
else:
    keyword_report = build_keyword_report(silver_v2.iter_records())
    keyword_report["source"] = {"silver_v2_report": SILVER_V2_REPORT_NAME, "silver_v2_sha256": verified_v2["sha256"]}
    KEYWORD_REPORT.parent.mkdir(parents=True, exist_ok=True)
    with KEYWORD_REPORT.open("x", encoding="utf-8", newline="\n") as handle:
        json.dump(keyword_report, handle, ensure_ascii=False, sort_keys=True, indent=2)
        handle.write("\n")
news_profile = keyword_report["classes"].get("News", {})
print("Keyword report:", KEYWORD_REPORT)
print("News rows:", news_profile.get("documents", 0))
print("Top News keywords:", news_profile.get("keywords", [])[:30])
print("News overlap:", keyword_report["news_vs_other_class_keyword_overlap"])
print("Examples:", news_profile.get("examples", []))

In [ ]:
PILOT_SAMPLE_SIZE = 300
TAXONOMY_SUFFIX = hashlib.sha256(json.dumps(get_v3_domain_labels()).encode("utf-8")).hexdigest()[:8]
NEWS_DECISIONS = v3_reports / f"news_review_decisions_v3_{source_suffix}_{TAXONOMY_SUFFIX}.jsonl"
NEWS_PILOT = v3_reports / f"news_pilot_v3_{source_suffix}_{TAXONOMY_SUFFIX}_n{PILOT_SAMPLE_SIZE}_seed42.jsonl"
if NEWS_PILOT.exists():
    print("Keeping existing pilot review file:", NEWS_PILOT)
else:
    pilot_rows = sample_news_records(silver_v2.iter_records(), sample_size=PILOT_SAMPLE_SIZE, seed=42)
    NEWS_PILOT.parent.mkdir(parents=True, exist_ok=True)
    with NEWS_PILOT.open("x", encoding="utf-8", newline="\n") as handle:
        for row in pilot_rows:
            item = {"id": row["id"], "text": str(row["text"]), "prompt": build_prompt(row, get_v3_domain_labels()), "pilot_only": True, "label": "", "needs_review": True, "approved": False, "reviewer": "", "reason": ""}
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
    print("Created pilot-only sample:", NEWS_PILOT, "| sampled rows:", len(pilot_rows))
print("Use this pilot to revise/freeze the taxonomy and guideline. It cannot publish Silver V3.")

## Optional local LLM suggestions (no API key)

Run this only after reviewing the pilot and finalizing the label guide. It downloads the public model weights, performs inference in this Colab GPU runtime, and writes unapproved suggestions to your mounted Drive; article text is not sent to an LLM API. The candidate file includes article text for review, so keep the Drive private and do not commit or share the file publicly. The source dataset license is marked unverified.

Enable a GPU in **Runtime → Change runtime type → GPU**. Inference resumes from checkpoints; `RESUME_LOCAL_MODEL_LABELING=True` is only for an unreviewed output from the same model/taxonomy. Long inputs are truncated to configured limits and explicitly marked for review.

In [ ]:
INSTALL_LOCAL_MODEL_DEPS = False  # Optional: install transformers/accelerate/bitsandbytes.
if INSTALL_LOCAL_MODEL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[local-llm]"], cwd=PROJECT_DIR, check=True)

RUN_LOCAL_MODEL_LABELING = False  # Keep False until the pilot and label guide are approved.
RESUME_LOCAL_MODEL_LABELING = False
LOCAL_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
if RUN_LOCAL_MODEL_LABELING:
    command = [
        sys.executable, "scripts/15_label_news_v3_local.py",
        "--data-lake-root", str(DATA_LAKE_ROOT),
        "--silver-v2-report", SILVER_V2_REPORT_NAME,
        "--output", str(NEWS_DECISIONS),
        "--model-name", LOCAL_MODEL_NAME,
        "--batch-size", "2",
    ]
    if RESUME_LOCAL_MODEL_LABELING:
        command.append("--resume")
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
else:
    print("Local LLM is optional and disabled. No API key is needed for this path.")

In [ ]:
CREATE_FULL_REVIEW_TEMPLATE = False  # Enable only after reviewing the pilot and freezing taxonomy.
if NEWS_DECISIONS.exists():
    print("Keeping existing review file; do not overwrite reviewed work:", NEWS_DECISIONS)
elif CREATE_FULL_REVIEW_TEMPLATE:
    with NEWS_DECISIONS.open("x", encoding="utf-8", newline="\n") as handle:
        for row in silver_v2.iter_records():
            if row["domain"] != "News":
                continue
            item = {"id": row["id"], "source_domain": "News", "text": row["text"], "prompt": build_prompt(row, get_v3_domain_labels()), "label": "", "needs_review": True, "approved": False, "reviewer": "", "reason": ""}
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
    print("Created full News review template:", NEWS_DECISIONS)
else:
    print("Pilot first. After finalizing the label guide, set CREATE_FULL_REVIEW_TEMPLATE=True and rerun this cell.")
print("Every full News row needs label/reviewer/reason, approved=true, needs_review=false.")
print("Candidate taxonomy:", get_v3_domain_labels())

In [ ]:
PUBLISH_SILVER_V3 = False  # Enable only after reviewing every full News decision.
silver_v3 = None
if not NEWS_DECISIONS.is_file():
    if PUBLISH_SILVER_V3:
        raise FileNotFoundError("Create the full News review file after finishing the pilot first")
    print("Silver V3 is not ready; create and review all News decisions first.")
else:
    silver_v3_pipeline = SilverV3Pipeline(silver_v2, NEWS_DECISIONS)
    decisions_sha256 = hashlib.sha256(NEWS_DECISIONS.read_bytes()).hexdigest()
    expected_silver = silver_v3_pipeline.planned_paths(verified_v2["sha256"], decisions_sha256)
    existing_silver = [path for path in expected_silver.values() if path.exists()]
    stale_silver = [path for path in expected_silver.values() if path.with_name(path.name + ".uploading").exists()]
    print("Silver V3 output:", {name: str(path) for name, path in expected_silver.items()})
    if stale_silver or (existing_silver and len(existing_silver) != len(expected_silver)):
        raise FileExistsError("Partial Silver V3 publication or stale upload; inspect Drive before retrying")
    if existing_silver:
        silver_v3 = SilverReader(DATA_LAKE_ROOT, expected_silver["report"].name)
        print("Existing Silver V3 verified:", SilverV3Pipeline.verify_reader(silver_v3))
    elif PUBLISH_SILVER_V3:
        silver_v3_pipeline.run()
        silver_v3 = SilverReader(DATA_LAKE_ROOT, expected_silver["report"].name)
        print("Silver V3 published and verified:", SilverV3Pipeline.verify_reader(silver_v3))
    else:
        print("No Silver V3 snapshot written. Review/edit decisions, then set PUBLISH_SILVER_V3=True.")

## Silver V3 outputs

- `02_silver/v3/`: immutable Silver V3 JSONL.
- `04_reports/silver/v3/`: keyword report, pilot, review decisions, audit, and quality report.

Do not treat the pilot or model suggestions as approved labels. The full News decision file must be complete and human-reviewed before enabling `PUBLISH_SILVER_V3`. The source dataset license is marked unverified; keep Drive access private.
